<a href="https://www.kaggle.com/code/rishabhbhartiya/top-500-crypto-coins-ohlc-data-indicators?scriptVersionId=289526228" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"
df = pd.read_csv("/kaggle/input/top-500-crypto-2025-historical-data/crypto_ohlc.csv")
df_1 = pd.read_csv("/kaggle/input/top-500-crypto-2025-historical-data/top_500_metadata.csv")

class AdvancedCryptoDashboard:
    def __init__(self, df_hist, df_meta):
        self.df = df_hist.copy()
        self.meta = df_meta.set_index("id")
        self.df["date"] = pd.to_datetime(self.df["date"])
        self.groups = {
            cid: g.sort_values("date")
            for cid, g in self.df.groupby("coin_id")
            if cid in self.meta.index
        }
        self.coin_ids = list(self.groups.keys())

    def _indicators(self, d):
        d = d.copy()
        d["ma20"] = d["close"].rolling(20).mean()
        d["ma50"] = d["close"].rolling(50).mean()
        d["returns"] = d["close"].pct_change()
        d["volatility"] = d["returns"].rolling(14).std()
        delta = d["close"].diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        rs = gain.rolling(14).mean() / loss.rolling(14).mean()
        d["rsi"] = 100 - (100 / (1 + rs))
        ema12 = d["close"].ewm(span=12, adjust=False).mean()
        ema26 = d["close"].ewm(span=26, adjust=False).mean()
        d["macd"] = ema12 - ema26
        d["macd_signal"] = d["macd"].ewm(span=9, adjust=False).mean()
        return d

    def _header_elements(self, m):
        annotations = []
        annotations.append(dict(
            x=0.01, y=1.36,
            xref="paper", yref="paper",
            text=f"<b>{m['name']}</b>",
            showarrow=False,
            align="left",
            font=dict(size=22)
        ))
        annotations.append(dict(
            x=0.16, y=1.24,
            xref="paper", yref="paper",
            text=f"<b>Symbol</b><br>{m['symbol'].upper()}",
            showarrow=False,
            align="left",
            bgcolor="rgba(30,30,30,0.95)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12)
        ))
        annotations.append(dict(
            x=0.32, y=1.24,
            xref="paper", yref="paper",
            text=f"<b>Market Cap</b><br>${m['market_cap']:,}",
            showarrow=False,
            align="left",
            bgcolor="rgba(30,30,30,0.95)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12)
        ))
        annotations.append(dict(
            x=0.52, y=1.24,
            xref="paper", yref="paper",
            text=f"<b>Total Volume</b><br>${m['total_volume']:,}",
            showarrow=False,
            align="left",
            bgcolor="rgba(30,30,30,0.95)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12)
        ))
        annotations.append(dict(
            x=0.75, y=1.24,
            xref="paper", yref="paper",
            text=f"<b>Rank</b><br>#{m['market_cap_rank']}",
            showarrow=False,
            align="left",
            bgcolor="rgba(45,45,45,0.95)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12)
        ))
        images = [dict(
            source=m["image"],
            x=0.01, y=1.28,
            xref="paper", yref="paper",
            sizex=0.07, sizey=0.07,
            xanchor="left",
            yanchor="top"
        )]
        return annotations, [], images

    def render(self):
        fig = make_subplots(
            rows=5,
            cols=1,
            shared_xaxes=True,
            row_heights=[0.45, 0.14, 0.14, 0.13, 0.14],
            vertical_spacing=0.035,
            subplot_titles=[
                "Price (OHLC + MA)",
                "Returns",
                "Volatility",
                "RSI (14)",
                "MACD"
            ]
        )
        traces_per_coin = 8
        buttons = []
        for i, cid in enumerate(self.coin_ids):
            d = self._indicators(self.groups[cid])
            m = self.meta.loc[cid]
            fig.add_trace(go.Candlestick(
                x=d["date"], open=d["open"], high=d["high"],
                low=d["low"], close=d["close"],
                visible=(i == 0), name="Price"
            ), 1, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["ma20"],
                                     visible=(i == 0), name="MA20"), 1, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["ma50"],
                                     visible=(i == 0), name="MA50"), 1, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["returns"],
                                     visible=(i == 0), name="Returns"), 2, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["volatility"],
                                     visible=(i == 0), name="Volatility"), 3, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["rsi"],
                                     visible=(i == 0), name="RSI"), 4, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["macd"],
                                     visible=(i == 0), name="MACD"), 5, 1)
            fig.add_trace(go.Scatter(x=d["date"], y=d["macd_signal"],
                                     visible=(i == 0), name="Signal"), 5, 1)
            vis = [False] * (len(self.coin_ids) * traces_per_coin)
            for j in range(traces_per_coin):
                vis[i * traces_per_coin + j] = True
            annotations, shapes, images = self._header_elements(m)
            buttons.append(dict(
                label=m["symbol"].upper(),
                method="update",
                args=[
                    {"visible": vis},
                    {
                        "annotations": annotations,
                        "shapes": shapes,
                        "images": images
                    }
                ]
            ))
        fig.update_layout(
            template="plotly_dark",
            height=1200,
            updatemenus=[dict(
                buttons=buttons,
                direction="down",
                x=0.01, y=1.50,
                bgcolor="rgba(20,20,20,0.95)",
                font=dict(size=11)
            )],
            xaxis=dict(dtick="M1", tickformat="%b %Y"),
            xaxis5=dict(dtick="M1", tickformat="%b %Y"),
            legend=dict(orientation="h")
        )
        fig.show()


In [2]:
dashboard = AdvancedCryptoDashboard(df, df_1)
dashboard.render()